In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn import set_config

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Dataset/loan_data.csv")

In [ ]:
df.head()

,Age,Gender,Married,Education,Employment,City,Dependents,Income,LoanAmount,CreditScore,TermMonths,Approved
0,25.0,Female,Yes,Graduate,Self-Employed,Bengaluru,0,49000.0,50000.0,672.0,84,N
1,55.0,Male,Yes,Postgraduate,Salaried,Mumbai,4,NaN,50000.0,705.0,24,Y
2,50.0,Male,Yes,NaN,Self-Employed,Mumbai,2,66300.0,50000.0,730.0,84,Y
3,40.0,Male,Yes,Graduate,Business,Pune,1,65000.0,50000.0,673.0,36,Y
4,40.0,Female,No,Postgraduate,Self-Employed,Bengaluru,2,68400.0,50000.0,756.0,60,Y


In [ ]:
df.isnull().sum()

,0
Age,48
Gender,0
Married,18
Education,36
Employment,42
City,24
Dependents,0
Income,60
LoanAmount,0
CreditScore,30


In [ ]:
# Split the data into train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns=['Approved']),df['Approved'],random_state=42,test_size=0.2)

In [ ]:
X_train.head()

,Age,Gender,Married,Education,Employment,City,Dependents,Income,LoanAmount,CreditScore,TermMonths
145,59.0,Female,Yes,Graduate,Salaried,Bengaluru,4,79000.0,50000.0,708.0,24
9,25.0,Male,Yes,Graduate,Salaried,Bengaluru,3,62400.0,50000.0,699.0,84
375,25.0,Male,Yes,Graduate,Salaried,Lucknow,3,26600.0,50000.0,765.0,36
523,58.0,Female,Yes,High School,Business,Bengaluru,2,56200.0,50000.0,764.0,24
188,31.0,Male,Yes,Graduate,Salaried,Delhi,1,55300.0,50000.0,636.0,24


In [ ]:
# Handel Missing Value
trf1=ColumnTransformer(transformers=[
    ('impute_Age_Income_CreditScore',SimpleImputer(),['Age','Income','CreditScore']),
    ('impute_Gender_Married_Education_Employment_City',SimpleImputer(strategy='most_frequent'),['Gender','Married','Education','Employment','City'])
],remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

In [ ]:
# Encoding
trf2=ColumnTransformer(transformers=[
    ('encoder_Gender_Married_Employment,City',OneHotEncoder(sparse_output=False,dtype=int),['Gender','Married','Employment','City']),
    ('encoder_Education',OrdinalEncoder(categories=[['High School','Graduate','Postgraduate']]),['Education'])
],verbose_feature_names_out=False,remainder='passthrough').set_output(transform='pandas')

In [ ]:
# Standardization
trf3=ColumnTransformer(transformers=[
    ('change_age_Income_Loan_credit_Month',StandardScaler(),['Age','Income','LoanAmount','CreditScore','TermMonths']),
],remainder='passthrough',verbose_feature_names_out=False
)

In [ ]:
pipeline=Pipeline([
    ('Missing Value',trf1),
    ('Encoding',trf2),
    ('Scale',trf3)
]
)


In [ ]:
# Architecture
set_config(display='diagram')

In [ ]:
pipeline

Pipeline(steps=[('Missing Value',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_Age_Income_CreditScore',
                                                  SimpleImputer(),
                                                  ['Age', 'Income',
                                                   'CreditScore']),
                                                 ('impute_Gender_Married_Education_Employment_City',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['Gender', 'Married',
                                                   'Education', 'Employment',
                                                   'City'])],
                                   verbose_feature_names_out=False)),
                (...
                                                 ('encoder_Education',
                                                  OrdinalEncoder(categories=[['High '
                                                                              'School',
                                                                              'Graduate',
                                                                              'Postgraduate']]),
                                                  ['Education'])],
                                   verbose_feature_names_out=False)),
                ('Scale',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('change_age_Income_Loan_credit_Month',
                                                  StandardScaler(),
                                                  ['Age', 'Income',
                                                   'LoanAmount', 'CreditScore',
                                                   'TermMonths'])],
                                   verbose_feature_names_out=False))])

In [ ]:
arr=pipeline.fit_transform(X_train,Y_train)

In [ ]:
pd.DataFrame(arr,columns=pipeline.get_feature_names_out())

,Age,Income,LoanAmount,CreditScore,TermMonths,Gender_Female,Gender_Male,Married_No,Married_Yes,Employment_Business,Employment_Salaried,Employment_Self-Employed,City_Bengaluru,City_Delhi,City_Lucknow,City_Mumbai,City_Pune,Education,Dependents
0,1.290091,0.732450,-0.365017,0.442275,-0.774089,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,4.0
1,-1.425624,0.083003,-0.365017,0.312902,1.516681,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0
2,-1.425624,-1.317611,-0.365017,1.261634,-0.315935,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,3.0
3,1.210217,-0.159561,-0.365017,1.247259,-0.774089,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
4,-0.946380,-0.194772,-0.365017,-0.592705,-0.774089,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,-1.106128,-0.417775,-0.365017,0.413525,-0.315935,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
476,1.050469,0.806784,2.497367,0.068532,0.600373,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0
477,-0.706758,-0.492110,-0.365017,-0.607080,-0.774089,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
478,-1.345750,-0.085227,-0.365017,0.126031,0.600373,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
